In [1]:
# Zelle 1: Bibliotheken laden. Alle weiteren Zellen benutzen diese Namen (pn, plf, np, pd, nx, plt).
import pypowsybl as pp  # Netzmodell und Lastfluss (pypowsybl)
import pypowsybl.network as pn  # Netze laden und ändern: pn.load(...), update_lines(...), get_buses(...)
import pypowsybl.loadflow as plf  # Lastflussrechnung: plf.run_ac(...), plf.run_dc(...)
import numpy as np  # Matrizen und Vektoren
from pypowsybl_jupyter import NetworkMapWidget, network_explorer  # interaktive Netzansicht im Notebook
import matplotlib.pyplot as plt  # Diagramme (Zelle 5)
import pandas as pd  # Tabellen
import math as m
import os
import networkx as nx  # Graphen: Inseln finden (Zelle 5 und 11)
import pandas as pd


In [2]:
# Zelle 2: KONFIGURATION. Nur hier muss etwas eingetragen werden, alle weiteren Zellen laufen dann einfach durch.
#
# NETZ = das Netz, das geprüft werden soll. Zwei Möglichkeiten:
#   a) Pfad zu einer Netzdatei (xiidm, uct, ...):  NETZ = "/home/.../mein_netz.xiidm"
#   b) pypowsybl-Testnetz (ohne Klammern!):        NETZ = pn.create_ieee14   (auch create_ieee30, create_ieee57, ...)
from pathlib import Path

NETZ = pn.create_ieee30
#NETZ = "/home/vc8596/ToOp/data/complex_grid/grid.xiidm"


# Es werden alle Ausfallmengen mit 1 bis MAX_AUSFAELLE Zweigen geprüft, die das Netz in mehr Inseln teilen.
# Die Rechenzeit wächst stark mit diesem Wert. Richtwert: complex_grid (30 Zweige) mit 4 dauert etwa 2 Minuten.
MAX_AUSFAELLE = 3

# Schutz gegen zu große Läufe: Wenn es mehr Kombinationen gäbe, bricht die Rechnung mit einer Meldung ab.
MAX_KOMBINATIONEN = 200_000

# Abweichungen unterhalb dieser Schwelle (in MW) gelten als übereinstimmend
TOL = 1e-6

# Für Zelle 6 (Tabelle Zweig für Zweig): Welche Ausfallmenge soll angesehen werden?
#   None = der Fall mit der größten Abweichung (im unten gewählten Slack-Modell)
#   oder eine Liste von Zweig-IDs, z.B. ["L2-3-1", "L3-4-1"] (die IDs stehen in Zelle 5 in der Spalte outaged_branches)
AUSFALL_ANSEHEN = None
# Mit welchem Slack-Modell? "einzelner Slack" oder "verteilter Slack"
SLACK_ANSEHEN = "einzelner Slack"

In [3]:
# Zelle 3: Hilfsfunktionen. Hier muss nichts geändert werden.
# Idee: Für jede Ausfallmenge, die das Netz in mehr Inseln teilt, wird compute_split_modf_main_grid mit einem
# pypowsybl-DC-Lastfluss auf dem wirklich aufgetrennten Netz verglichen.
# 3-Wicklungs-Trafos werden als Stern modelliert: je Wicklung ein Zweig "<trafo>__leg1" bis "__leg3" und ein
# zusätzlicher Sternpunkt-Bus "<trafo>__star".
import itertools
from collections.abc import Callable
from math import comb

import jax
import jax.numpy as jnp
from toop_engine_dc_solver.jax.split_modf_main_grid import compute_split_modf_main_grid  # die zu testende Funktion

jax.config.update("jax_enable_x64", True)  # doppelte Genauigkeit, sonst stimmt der Vergleich mit pypowsybl nicht

BRANCH_COLUMNS = ["bus1_id", "bus2_id", "connected1", "connected2"]
# Slack-Modell 1: ein einziger Slack-Bus gleicht die Bilanz aus (so rechnet auch compute_split_modf_main_grid)
SINGLE_SLACK = plf.Parameters(distributed_slack=False)
# Slack-Modell 2: der Leistungsfehler wird anteilig auf alle Generatoren verteilt (Standard im Produktions-Runner)
DISTRIBUTED_SLACK = plf.Parameters(distributed_slack=True, balance_type=plf.BalanceType.PROPORTIONAL_TO_GENERATION_P)

LEG = "__leg"  # Trenner in der Zweig-ID einer Wicklung: "<trafo>__leg1" ... "<trafo>__leg3"
STAR = "__star"  # Endung des künstlichen Sternpunkt-Busses eines 3-Wicklungs-Trafos
TWT_COLUMNS = ["bus1_id", "bus2_id", "bus3_id", "connected1", "connected2", "connected3"]


def make_factory(netz) -> Callable[[], pn.Network]:
    """Macht aus NETZ (Pfad oder pypowsybl-Funktion) eine Funktion, die bei jedem Aufruf ein frisches Netz liefert."""
    if callable(netz):
        return netz
    path = Path(netz)
    return lambda: pn.load(path)


def _three_winding_legs(net: pn.Network) -> pd.DataFrame:
    """Zerlegt jeden 3-Wicklungs-Trafo in drei Zweige (Anschluss-Bus -> Sternpunkt-Bus)."""
    rows = []
    for trafo_id, trafo in net.get_3_windings_transformers(attributes=TWT_COLUMNS).iterrows():
        for leg in (1, 2, 3):
            rows.append(
                {
                    "id": f"{trafo_id}{LEG}{leg}",
                    "bus1_id": trafo[f"bus{leg}_id"],
                    "bus2_id": f"{trafo_id}{STAR}",
                    "connected1": trafo[f"connected{leg}"],
                    "connected2": True,
                    "trafo": trafo_id,
                    "leg": leg,
                }
            )
    return pd.DataFrame(rows, columns=[*BRANCH_COLUMNS, "id", "trafo", "leg"]).set_index("id")


def _branch_flows(net: pn.Network) -> pd.Series:
    """Wirkleistung am Anfang aller Leitungen und 2W-Trafos und in jede Wicklung der 3W-Trafos hinein."""
    twt = net.get_3_windings_transformers(attributes=["p1", "p2", "p3"])
    legs = [twt[f"p{leg}"].rename(lambda trafo_id, leg=leg: f"{trafo_id}{LEG}{leg}") for leg in (1, 2, 3)]
    return pd.concat(
        [net.get_lines(attributes=["p1"]).p1, net.get_2_windings_transformers(attributes=["p1"]).p1, *legs]
    )


def _run_dc(net: pn.Network, params: plf.Parameters, slack_voltage_level: str | None = None) -> None:
    """DC-Lastfluss rechnen, auch wenn das Netz in mehrere Inseln zerfällt (ALL_CONNECTED = alle Inseln rechnen).

    Mit slack_voltage_level wird der Slack auf diese Spannungsebene festgelegt. Eine Bus-ID taugt dafür nicht, weil
    pypowsybl die Busse nach dem Öffnen von Zweigen umnummeriert und sonst still einen anderen Slack wählt.
    """
    provider = {"slackBusSelectionMode": "NAME", "slackBusesIds": slack_voltage_level} if slack_voltage_level else {}
    all_components = plf.Parameters(
        distributed_slack=params.distributed_slack,
        balance_type=params.balance_type,
        component_mode=plf.ComponentMode.ALL_CONNECTED,
        provider_parameters=provider,
    )
    plf.run_dc(net, all_components)


def _dc_state(net: pn.Network, params: plf.Parameters, branches: pd.DataFrame) -> tuple[np.ndarray, pd.Series]:
    """DC-Lastfluss rechnen und zurückgeben: Zweigflüsse und Winkel (rad) der echten Busse."""
    _run_dc(net, params)
    angles = np.deg2rad(net.get_buses().v_angle)
    return _branch_flows(net).loc[branches.index].to_numpy(), angles


def _estimate_susceptances(
    d_flow: np.ndarray, d_theta: np.ndarray, branches: pd.DataFrame, from_node: np.ndarray, to_node: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """Schätzt je Zweig die Suszeptanz b aus Flussänderungen (Zeilen = DC-Läufe) und Winkeländerungen der Busse.

    Leitungen und 2W-Trafos: Flussänderung = b * Winkeldifferenz. Bei einer Wicklung eines 3W-Trafos ist der
    Winkel des Sternpunkts unbekannt. Aus Flussänderung_i = b_i * (Winkel_i - Winkel_Stern) folgt aber
    x_i * Flussänderung_i - x_j * Flussänderung_j = Winkel_i - Winkel_j mit x = 1 / b. Das ist linear in x.
    Rückgabe: Suszeptanzen und die Winkeldifferenz über jedem Zweig (bei Wicklungen inklusive Sternpunkt-Winkel).
    """
    susceptance = np.full(len(branches), np.nan)
    d_angle = np.full(d_flow.shape, np.nan)
    ordinary = branches.trafo.isna().to_numpy()
    d_angle[:, ordinary] = d_theta[:, from_node[ordinary]] - d_theta[:, to_node[ordinary]]
    susceptance[ordinary] = (d_flow[:, ordinary] * d_angle[:, ordinary]).sum(0) / (d_angle[:, ordinary] ** 2).sum(0)

    for _trafo_id, group in branches[~ordinary].groupby("trafo"):
        legs = [branches.index.get_loc(leg_id) for leg_id in group.sort_values("leg").index]
        if len(legs) != 3:
            raise ValueError("Ein 3W-Trafo mit abgeschalteter Wicklung wird nicht unterstützt.")
        theta = [d_theta[:, from_node[leg]] for leg in legs]
        # Zeilen: je DC-Lauf die Paare (Wicklung 1, 2) und (Wicklung 1, 3); Unbekannte: x_1, x_2, x_3
        zero = np.zeros(d_flow.shape[0])
        matrix = np.vstack(
            [
                np.column_stack([d_flow[:, legs[0]], -d_flow[:, legs[1]], zero]),
                np.column_stack([d_flow[:, legs[0]], zero, -d_flow[:, legs[2]]]),
            ]
        )
        target = np.concatenate([theta[0] - theta[1], theta[0] - theta[2]])
        reactance = np.linalg.lstsq(matrix, target, rcond=None)[0]
        b = 1.0 / reactance
        susceptance[legs] = b
        # Sternpunkt-Winkel aus der Knotenregel: sum(b_i * (Winkel_i - Winkel_Stern)) = 0
        theta_star = sum(b[i] * theta[i] for i in range(3)) / b.sum()
        for i, leg in enumerate(legs):
            d_angle[:, leg] = theta[i] - theta_star
    return susceptance, d_angle


def _build_model(factory: Callable[[], pn.Network], params: plf.Parameters) -> dict:
    """Baut aus dem intakten Netz alles, was compute_split_modf_main_grid braucht (PTDF, Basisflüsse, Einspeisungen).

    3-Wicklungs-Trafos werden als Stern modelliert: drei Zweige und ein zusätzlicher Sternpunkt-Bus je Trafo.
    """
    net = factory()
    # Alle Zweige sammeln, die an beiden Enden verbunden sind (Wicklungen von 3W-Trafos sind eigene Zweige)
    branches = pd.concat(
        [
            net.get_lines(attributes=BRANCH_COLUMNS),
            net.get_2_windings_transformers(attributes=BRANCH_COLUMNS),
            _three_winding_legs(net),
        ]
    )
    branches = branches[branches.connected1 & branches.connected2]
    # Echte Busse zuerst, danach je 3W-Trafo ein Sternpunkt-Bus
    real_buses = net.get_buses().index
    stars = pd.Index(sorted(set(branches.bus2_id[branches.trafo.notna()])))
    buses = real_buses.append(stars)
    from_node = buses.get_indexer(branches.bus1_id)
    to_node = buses.get_indexer(branches.bus2_id)
    if (from_node < 0).any() or (to_node < 0).any():
        raise ValueError("Ein Zweig hängt an einem Bus, der in get_buses() fehlt (z.B. Netz mit Knoten-Topologie).")
    # Zweig-Knoten-Inzidenzmatrix (+1 am Anfangsbus, -1 am Endbus)
    incidence = np.zeros((len(branches), len(buses)))
    incidence[np.arange(len(branches)), from_node] = 1.0
    incidence[np.arange(len(branches)), to_node] = -1.0

    # Ein erster DC-Lauf liefert den Slack-Bus von pypowsybl und die Referenzwerte für die Suszeptanz-Schätzung
    results = plf.run_dc(net, SINGLE_SLACK)
    slack = buses.get_loc(results[0].reference_bus_id)
    fit_flow, fit_angle = _dc_state(net, SINGLE_SLACK, branches)
    # Basisflüsse n_0_flow mit dem gewählten Slack-Modell (params). Sie sind der Normalzustand vor dem Ausfall.
    _run_dc(net, params)
    n_0_flow = _branch_flows(net).loc[branches.index].to_numpy()

    # Suszeptanzen schätzen: 6 DC-Läufe mit zufällig veränderten Einspeisungen (-50 % bis +50 %).
    # Phasenschieber-Versätze heben sich in den Differenzen auf.
    rng = np.random.default_rng(0)
    d_flow, d_theta = [], []
    for _ in range(6):
        perturbed = factory()
        gens, loads = perturbed.get_generators(), perturbed.get_loads()
        perturbed.update_generators(
            id=gens.index.tolist(), target_p=(gens.target_p * rng.uniform(0.5, 1.5, len(gens))).tolist()
        )
        perturbed.update_loads(id=loads.index.tolist(), p0=(loads.p0 * rng.uniform(0.5, 1.5, len(loads))).tolist())
        flow, angle = _dc_state(perturbed, SINGLE_SLACK, branches)
        d_flow.append(flow - fit_flow)
        d_theta.append(angle.reindex(real_buses).to_numpy() - fit_angle.reindex(real_buses).to_numpy())
    d_flow, d_theta = np.array(d_flow), np.array(d_theta)
    # Winkel gibt es nur für echte Busse: Sternpunkt-Spalten mit NaN auffüllen
    d_theta = np.pad(d_theta, ((0, 0), (0, len(stars))), constant_values=np.nan)
    susceptance, d_angle = _estimate_susceptances(d_flow, d_theta, branches, from_node, to_node)
    # Kontrolle: Passt "Flussänderung = b * Winkeländerung" auf allen Zweigen? Sonst wäre die PTDF falsch.
    if not np.all(np.isfinite(susceptance)) or np.abs(d_flow - susceptance * d_angle).max() > 1e-6:
        raise ValueError("Die Suszeptanzen lassen sich nicht aus den DC-Läufen schätzen (z.B. Zweig ohne Winkeldifferenz).")

    # PTDF = diag(b) * A * B_red^-1; die Spalte des Slack-Busses bleibt 0 (Winkelreferenz)
    keep = np.arange(len(buses)) != slack
    bus_matrix = incidence.T @ (susceptance[:, None] * incidence)
    ptdf = np.zeros_like(incidence)
    ptdf[:, keep] = (susceptance[:, None] * incidence[:, keep]) @ np.linalg.inv(bus_matrix[np.ix_(keep, keep)])
    # Rückgabe: Zweige, Anzahl Busse, Zweigenden, Slack, Basisflüsse, Einspeisung je Bus (= A^T * Flüsse) und PTDF
    return {
        "branches": branches,
        "n_buses": len(buses),
        "n_real_buses": len(real_buses),
        "from_node": from_node,
        "to_node": to_node,
        "slack": slack,
        "slack_voltage_level": net.get_buses(attributes=["voltage_level_id"]).voltage_level_id.iloc[slack],
        "n_0_flow": n_0_flow,
        "nodal_injection": incidence.T @ n_0_flow,
        "ptdf": ptdf,
    }


def _island_creating_outages(model: dict, max_outages: int) -> list[np.ndarray]:
    """Findet alle Ausfallmengen mit bis zu max_outages Zweigen, die die Zahl der elektrischen Inseln erhöhen."""
    edges = list(zip(model["from_node"].tolist(), model["to_node"].tolist(), strict=True))
    n_combinations = sum(comb(len(edges), k) for k in range(1, max_outages + 1))
    if n_combinations > MAX_KOMBINATIONEN:
        raise ValueError(
            f"{n_combinations} Kombinationen sind mehr als MAX_KOMBINATIONEN={MAX_KOMBINATIONEN}. "
            "MAX_AUSFAELLE verkleinern oder MAX_KOMBINATIONEN erhöhen."
        )
    star_nodes = range(model["n_real_buses"], model["n_buses"])

    # Zählt die Inseln (zusammenhängende Teilnetze), wenn die Zweige in 'removed' fehlen.
    # Ein Sternpunkt ohne verbleibende Wicklung ist keine Insel (der Trafo ist dann nur ganz abgeschaltet).
    def n_components(removed: set[int]) -> int:
        graph = nx.Graph()
        graph.add_nodes_from(range(model["n_buses"]))
        graph.add_edges_from(edge for index, edge in enumerate(edges) if index not in removed)
        graph.remove_nodes_from([star for star in star_nodes if graph.degree(star) == 0])
        return nx.number_connected_components(graph)

    # Alle Kombinationen aus 1 bis max_outages Zweigen probieren; behalten werden nur die, die mehr Inseln erzeugen
    intact = n_components(set())
    return [
        np.array(outage_set)
        for n_outages in range(1, max_outages + 1)
        for outage_set in itertools.combinations(range(len(edges)), n_outages)
        if n_components(set(outage_set)) > intact
    ]


def _open_branches(net: pn.Network, branch_ids: list[str]) -> None:
    """Schaltet Zweige ab: Leitungen und 2W-Trafos beidseitig, bei 3W-Trafos nur die genannte Wicklung.

    Bei Netzen mit Schaltern (z.B. CGMES) greift 'connected=False' nicht immer, ohne dass pypowsybl einen Fehler
    meldet. Deshalb wird geprüft, ob der Zweig wirklich offen ist. Sonst werden Trenner betätigt und zuletzt wird
    der Zweig entfernt. Eine Wicklung, die nicht abgeschaltet werden kann, bricht mit einem Fehler ab.
    """
    lines = set(net.get_lines().index)
    for branch_id in branch_ids:
        if LEG in branch_id:
            trafo_id, leg = branch_id.rsplit(LEG, 1)
            net.update_3_windings_transformers(id=trafo_id, **{f"connected{leg}": False})
            if net.get_3_windings_transformers(attributes=[f"connected{leg}"]).loc[trafo_id].iloc[0]:
                raise ValueError(f"Wicklung {branch_id} lässt sich nicht abschalten.")
            continue
        is_line = branch_id in lines
        getter = net.get_lines if is_line else net.get_2_windings_transformers

        def is_connected(branch_id: str = branch_id, getter: Callable = getter) -> bool:
            return bool(getter(attributes=["connected1", "connected2"]).loc[branch_id].any())

        (net.update_lines if is_line else net.update_2_windings_transformers)(
            id=branch_id, connected1=False, connected2=False
        )
        if is_connected():
            net.disconnect(branch_id, operate_disconnectors=True, operate_fictitious=True)
        if is_connected():
            net.remove_elements([branch_id])


def compare_with_powsybl(factory: Callable[[], pn.Network], params: plf.Parameters, max_outages: int) -> pd.DataFrame:
    """Vergleicht split MODF mit pypowsybl DC auf der behaltenen Insel für jede inselbildende Ausfallmenge.

    Liefert eine Zeile je Ausfallmenge. Zusätzlich steht in discarded_imbalance_mw die Leistungsbilanz der
    abgetrennten Inseln, die compute_split_modf_main_grid verwirft und am Slack-Bus ausgleicht.
    Ausfallmengen, für die pypowsybl keine brauchbare Referenz liefert, werden übersprungen und stehen in
    df.attrs["skipped"] (Zweig nicht abschaltbar, oder behaltene Insel ohne Generator: NO_CALCULATION).
    """
    model = _build_model(factory, params)
    branches = model["branches"]
    outage_sets = _island_creating_outages(model, max_outages)
    if not outage_sets:
        raise ValueError("Keine Ausfallmenge teilt das Netz in mehr Inseln (z.B. ein vollständig vermaschtes Netz).")

    # Ausfallmengen unterschiedlicher Größe werden mit -1 auf die gleiche Länge aufgefüllt (-1 = kein Zweig)
    split_flow, _main_bus, main_monitored, imbalance, _rank_loss = compute_split_modf_main_grid(
        ptdf=jnp.asarray(model["ptdf"]),
        from_node=jnp.asarray(model["from_node"], dtype=jnp.int32),
        to_node=jnp.asarray(model["to_node"], dtype=jnp.int32),
        nodal_injections=jnp.asarray(model["nodal_injection"][None, :]),
        n_0_flow=jnp.asarray(model["n_0_flow"][None, :]),
        outages=jnp.asarray(
            np.stack([np.pad(s, (0, max_outages - len(s)), constant_values=-1) for s in outage_sets]), dtype=jnp.int32
        ),
        branches_monitored=jnp.arange(len(branches), dtype=jnp.int32),
        slack_bus=jnp.asarray(model["slack"], dtype=jnp.int32),
    )
    # split_flow[0] = Flüsse (Zeitschritt 0); NaN bei Zweigen außerhalb der behaltenen Insel.
    # main_monitored = Maske der Zweige, die in der behaltenen Insel liegen.
    split_flow = np.asarray(split_flow[0])
    main_monitored = np.asarray(main_monitored)
    discarded_imbalance = np.abs(np.asarray(imbalance)[0])

    # Je Ausfallmenge: frisches Netz laden, Zweige öffnen, DC rechnen, Fluss als Referenz nehmen
    rows, skipped = [], []
    for case, outage_set in enumerate(outage_sets):
        outaged_ids = branches.index[outage_set].tolist()
        cut_net = factory()
        try:
            _open_branches(cut_net, outaged_ids)
        except ValueError:
            skipped.append({"outaged_branches": ", ".join(outaged_ids), "grund": "Zweig nicht abschaltbar"})
            continue
        _run_dc(cut_net, params, model["slack_voltage_level"])
        reference = _branch_flows(cut_net).reindex(branches.index).to_numpy()
        # Ausgefallene Zweige haben Fluss 0 (pypowsybl liefert dort NaN)
        reference[outage_set] = 0.0
        if np.isnan(reference[main_monitored[case]]).any():
            skipped.append({"outaged_branches": ", ".join(outaged_ids), "grund": "pypowsybl rechnet die Insel nicht"})
            continue
        # Abweichung nur auf Zweigen der behaltenen Insel, dort muss split MODF mit pypowsybl übereinstimmen
        difference = np.abs(split_flow[case] - reference)[main_monitored[case]]
        max_diff = np.max(difference, initial=0.0)
        rows.append(
            {
                "outaged_branches": ", ".join(outaged_ids),
                "n_outages": len(outaged_ids),
                "n_kept_branches": int(main_monitored[case].sum()),
                "max_diff_mw": max_diff,
                "matches": bool(max_diff <= TOL),
                "discarded_imbalance_mw": discarded_imbalance[case],
            }
        )
    result = pd.DataFrame(rows)
    result.attrs["skipped"] = pd.DataFrame(skipped, columns=["outaged_branches", "grund"])
    return result

In [4]:
# Zelle 4: Prüfung ausführen. Hier muss nichts geändert werden (Netz und Größe stehen in Zelle 2).
# Es werden beide Slack-Modelle gerechnet: einzelner Slack (so rechnet die Funktion) und verteilter Slack.
factory = make_factory(NETZ)
slack_models = {"einzelner Slack": SINGLE_SLACK, "verteilter Slack": DISTRIBUTED_SLACK}
comparisons = {name: compare_with_powsybl(factory, params, MAX_AUSFAELLE) for name, params in slack_models.items()}

# Zusammenfassung: eine Zeile je Slack-Modell
# - Ausfallmengen: wie viele Kombinationen das Netz in mehr Inseln teilen (nur diese werden geprüft)
# - abweichend: bei wie vielen davon die Funktion von pypowsybl abweicht (mehr als TOL)
# - größte Abweichung: in MW, nur auf den Zweigen der Insel mit dem Slack-Bus
# - übersprungen: Ausfallmengen ohne brauchbare pypowsybl-Referenz (Gründe in comparisons[...].attrs["skipped"])
pd.DataFrame(
    [
        {
            "Slack-Modell": name,
            "Ausfallmengen": len(df),
            "abweichend": int((~df.matches).sum()),
            "größte Abweichung [MW]": df.max_diff_mw.max(),
            "übersprungen": len(df.attrs["skipped"]),
        }
        for name, df in comparisons.items()
    ]
).set_index("Slack-Modell")

,Ausfallmengen,abweichend,größte Abweichung [MW]
Slack-Modell,,,
einzelner Slack,3302,0,3.375078e-12
verteilter Slack,3302,1711,1.558961e+01


In [5]:
# Zelle 5: Details zu den Abweichungen beim verteilten Slack.
# Ursache: compute_split_modf_main_grid wirft die Einspeisungen der abgetrennten Inseln weg und legt die übrige
# Bilanz (discarded_imbalance_mw) auf den ursprünglichen Slack-Bus. pypowsybl verteilt sie dagegen auf die Generatoren.
# Kreuztabelle: Zeilen = wurde eine Bilanz verworfen?, Spalten = stimmt das Ergebnis mit pypowsybl überein?
distributed = comparisons[SLACK_ANSEHEN]
display(pd.crosstab(distributed.discarded_imbalance_mw > 1e-6, distributed.matches, rownames=["verworfene Bilanz"]))
# Die 10 größten Abweichungen beim verteilten Slack
distributed.sort_values("max_diff_mw", ascending=False).head(10)

matches,True
verworfene Bilanz,
False,1512
True,1790


,outaged_branches,n_outages,n_kept_branches,max_diff_mw,matches,discarded_imbalance_mw
1481,"L6-8-1, L9-11-1, L6-28-1",3,40,3.375078e-12,True,1.776357e-14
1513,"L6-8-1, L12-13-1, L6-28-1",3,40,3.357314e-12,True,1.065814e-14
1572,"L6-8-1, L25-26-1, L6-28-1",3,40,3.218759e-12,True,3.500000e+00
234,"L1-2-1, L4-6-1, L9-11-1",3,40,2.387424e-12,True,1.776357e-14
236,"L1-2-1, L4-6-1, L25-26-1",3,40,2.302158e-12,True,3.500000e+00
235,"L1-2-1, L4-6-1, L12-13-1",3,40,2.245315e-12,True,1.065814e-14
299,"L1-2-1, L12-13-1, L25-27-1",3,40,1.961098e-12,True,1.065814e-14
340,"L1-2-1, L25-26-1, L25-27-1",3,40,1.932676e-12,True,3.500000e+00
297,"L1-2-1, L12-13-1, L24-25-1",3,40,1.904255e-12,True,1.065814e-14
337,"L1-2-1, L24-25-1, L25-26-1",3,40,1.875833e-12,True,3.500000e+00


In [6]:
# Zelle 6: Tabelle Zweig für Zweig für EINE Ausfallmenge (Einstellung AUSFALL_ANSEHEN und SLACK_ANSEHEN in Zelle 2).
# Jede Zeile ist ein Zweig (Leitung oder Trafo). Die Spalten stellen die Flüsse gegenüber:
#   vor Ausfall   = Fluss im intakten Netz
#   Split MODF    = Fluss nach dem Ausfall laut compute_split_modf_main_grid (NaN = Zweig nicht in der behaltenen Insel)
#   pypowsybl     = Fluss nach dem Ausfall laut pypowsybl-DC-Lastfluss auf dem wirklich aufgetrennten Netz
#   Differenz     = Split MODF minus pypowsybl
slack_params = slack_models[SLACK_ANSEHEN]
if AUSFALL_ANSEHEN is None:
    # Fall mit der größten Abweichung aus Zelle 4 nehmen
    worst_case = comparisons[SLACK_ANSEHEN].sort_values("max_diff_mw", ascending=False).iloc[0]
    ausfall_ids = worst_case.outaged_branches.split(", ")
else:
    ausfall_ids = list(AUSFALL_ANSEHEN)
print("Ausfallmenge:", ausfall_ids, "| Slack-Modell:", SLACK_ANSEHEN)

# Modell des intakten Netzes (Basisflüsse, PTDF, Einspeisungen) und Index der ausgefallenen Zweige
view_model = _build_model(factory, slack_params)
view_branches = view_model["branches"]
view_outage = view_branches.index.get_indexer(ausfall_ids)
if (view_outage < 0).any():
    raise ValueError(f"Unbekannte Zweig-ID in {ausfall_ids}. Erlaubt sind die Zweige aus view_branches.index.")

# Split MODF für diese eine Ausfallmenge rechnen
view_flow, _, view_main_monitored, view_imbalance, _ = compute_split_modf_main_grid(
    ptdf=jnp.asarray(view_model["ptdf"]),
    from_node=jnp.asarray(view_model["from_node"], dtype=jnp.int32),
    to_node=jnp.asarray(view_model["to_node"], dtype=jnp.int32),
    nodal_injections=jnp.asarray(view_model["nodal_injection"][None, :]),
    n_0_flow=jnp.asarray(view_model["n_0_flow"][None, :]),
    outages=jnp.asarray(view_outage[None, :], dtype=jnp.int32),
    branches_monitored=jnp.arange(len(view_branches), dtype=jnp.int32),
    slack_bus=jnp.asarray(view_model["slack"], dtype=jnp.int32),
)

# Referenz: pypowsybl-DC-Lastfluss auf dem Netz mit geöffneten Zweigen (ausgefallene Zweige = 0)
view_cut_net = factory()
_open_branches(view_cut_net, ausfall_ids)
_run_dc(view_cut_net, slack_params, view_model["slack_voltage_level"])
view_reference = _branch_flows(view_cut_net).reindex(view_branches.index).to_numpy()
view_reference[view_outage] = 0.0

print(f"Verworfene Leistungsbilanz der abgetrennten Inseln: {float(view_imbalance[0, 0]):.2f} MW")
view_table = pd.DataFrame(
    {
        "von_bus": view_branches.bus1_id,
        "nach_bus": view_branches.bus2_id,
        "ausgefallen": view_branches.index.isin(ausfall_ids),
        "in behaltener Insel": np.asarray(view_main_monitored[0]),
        "vor Ausfall [MW]": view_model["n_0_flow"],
        "Split MODF [MW]": np.asarray(view_flow[0, 0]),
        "pypowsybl [MW]": view_reference,
    },
    index=view_branches.index,
)
view_table["Differenz [MW]"] = view_table["Split MODF [MW]"] - view_table["pypowsybl [MW]"]
view_table.round(3)

Ausfallmenge: ['L6-8-1', 'L9-11-1', 'L6-28-1'] | Slack-Modell: einzelner Slack
Verworfene Leistungsbilanz der abgetrennten Inseln: 0.00 MW


,von_bus,nach_bus,ausgefallen,in behaltener Insel,vor Ausfall [MW],Split MODF [MW],pypowsybl [MW],Differenz [MW]
id,,,,,,,,
L1-2-1,VL1_0,VL2_0,False,True,161.026,159.913,159.913,-0.0
L1-3-1,VL1_0,VL3_0,False,True,82.374,83.487,83.487,-0.0
L2-4-1,VL2_0,VL4_0,False,True,42.488,44.158,44.158,-0.0
L3-4-1,VL3_0,VL4_0,False,True,79.974,81.087,81.087,0.0
L2-5-1,VL2_0,VL5_0,False,True,77.970,77.113,77.113,-0.0
L2-6-1,VL2_0,VL6_0,False,True,58.868,56.941,56.941,-0.0
L4-6-1,VL4_0,VL6_0,False,True,72.424,57.208,57.208,-0.0
L5-7-1,VL5_0,VL7_0,False,True,-16.230,-17.087,-17.087,-0.0
L6-7-1,VL6_0,VL7_0,False,True,39.030,39.887,39.887,0.0
